# Lab 2.1 — Open-Source LLMs: First Contact
**Module II · LLMs & GNNs for Advanced Reasoning over Relational Data**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DanielFPerez/llm-gnns-course_solutions/blob/main/module-2-llm/lab2_1_open_source_llms.ipynb)

---

## What you will do
1. Load a local open-source LLM through a unified interface that works both locally (Ollama) and on Colab (HuggingFace fallback).
2. Generate text and understand what makes LLMs powerful at language tasks.
3. Use **system prompts** to shape the model's persona, style, and constraints — and understand their limits.
4. Observe how **temperature** controls creativity vs. consistency.
5. Deliberately trigger a **hallucination** by asking the LLM about information it cannot know — and understand *why* this happens structurally.
6. `[Extension]` Understand **context windows** and token counting.

## Prerequisites
Module I completed (or equivalent Python comfort). No prior LLM experience needed.

**Estimated time:** 50–65 min

---
## 0 · Setup

If you are running **locally with Ollama**: make sure Ollama is running (`ollama serve`) and that you have pulled a model:
```bash
ollama pull llama3.2:1b
```

If you are on **Google Colab** or Ollama is not running: the LLM wrapper will automatically download a small (~3 GB) HuggingFace model. This takes a few minutes the first time but is then cached.

### Using Ollama from the terminal (optional)

If you want to interact with the model directly from a terminal — without running any Python — Ollama provides its own CLI and a local REST API. Everything below is **optional**: the notebook works the same whether you use this or not.

---

#### 1 · Install Ollama

Download and run the installer for your OS from **[ollama.com/download](https://ollama.com/download)**.  
On Linux / WSL2 you can also run:
```bash
curl -fsSL https://ollama.com/install.sh | sh
```

---

#### 2 · Pull a model

```bash
ollama pull llama3.2:1b        # ~1.3 GB — fast to download, runs on CPU
ollama pull llama3.2:3b        # ~2.0 GB — better quality, still lightweight
```

List what you have downloaded:
```bash
ollama list
```

---

#### 3 · Start the server

The server must be running before the notebook (or any API call) can use it:
```bash
ollama serve
```
It listens on `http://localhost:11434`. Leave this terminal open while you work.

---

#### 4 · Interactive chat in the terminal

Open a **second** terminal and start a live chat session:
```bash
ollama run llama3.2:1b
```
Type your message and press Enter. Type `/bye` or Ctrl-D to exit.

```
>>> Explain what a transformer is in one sentence.
A transformer is a deep learning model architecture that uses self-attention ...
>>> /bye
```

---

#### 5 · One-shot generation via the REST API

You can also call the model with a single `curl` command — useful for scripting or quick tests:
```bash
curl http://localhost:11434/api/generate \
  -d '{
    "model": "llama3.2:1b",
    "prompt": "What is the difference between precision and recall?",
    "stream": false
  }' | python3 -m json.tool
```

For chat-style messages with a system prompt:
```bash
curl http://localhost:11434/api/chat \
  -d '{
    "model": "llama3.2:1b",
    "stream": false,
    "messages": [
      {"role": "system",  "content": "Answer in one sentence."},
      {"role": "user",    "content": "What is gradient descent?"}
    ]
  }' | python3 -m json.tool
```

The response field you want is `response` (for `/api/generate`) or `message.content` (for `/api/chat`).

---

> **Tip:** `SimpleLLM()` in this notebook calls exactly these two endpoints under the hood.  
> If `ollama serve` is not running, it automatically falls back to a HuggingFace model instead.

In [1]:
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    print("Running in Google Colab. Cloning the course solutions repository and installing dependencies...")
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/DanielFPerez/llm-gnns-course_solutions.git"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-r",
         "llm-gnns-course_solutions/environment/requirements.txt"], check=True)
    sys.path.insert(0, "llm-gnns-course_solutions")
else:
    sys.path.insert(0, str(Path("..").resolve()))

print("Setup complete.")

Setup complete.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd() / "MGCD_M_2026_Inter"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Proyecto añadido a sys.path:", PROJECT_ROOT)

Proyecto añadido a sys.path: /home/sylphild/bioforja/bioforja-projects/Cursos/MGCD_M_2026_Inter


In [7]:

from utils.llm import SimpleLLM
from utils import load_company_kb
print("Imports OK.")

Imports OK.


---
## 1 · The LLM wrapper

We use a thin `SimpleLLM` class that hides whether you are talking to Ollama or a HuggingFace model. The API is the same either way:

```python
llm.generate(prompt)     # single user message → reply
llm.chat(messages)       # list of {role, content} dicts → reply
```

The cell below creates the wrapper and auto-detects the backend.

In [8]:
llm = SimpleLLM(ollama_model='gemma3:4b')#backend='ollama', model='qwen2.5-coder:7b', temperature=0.2, max_tokens=512)
print(llm)

[SimpleLLM] Ollama detected → using model 'gemma3:4b'
            Tip: run `ollama pull gemma3:4b` if not yet downloaded.
SimpleLLM(backend='ollama', model='gemma3:4b')


In [ ]:
curl http://localhost:11434/api/chat \
  -H "Content-Type: application/json" \
  -d '{
    "model": "gemma3:4b",
    "messages": [
      {"role": "user", "content": "Explain machine learning in simple terms"}
    ],
    "stream": false
  }'

---
## 2 · Basic generation

### Exercise 2.1.1 `[Core]` — Hello, LLM

Ask the LLM to explain what machine learning is, in one sentence, as if speaking to a business executive with no technical background. Use `llm.generate(prompt)` and print the response.

In [9]:
message = [
    {"role": "system", "content": "You are a helpful assistant expert in machine learning"},
    {"role": "user", "content": "Explain what is machine learning in simple terms"},
]
llm.chat(message)

'Okay, let\'s break down machine learning in a way that’s easy to understand.\n\n**Essentially, Machine Learning is about teaching computers to learn *without* being explicitly programmed for every single situation.** \n\nHere\'s a breakdown using an analogy:\n\n**Imagine you\'re trying to teach a dog a trick like "fetch."**\n\n* **Traditional Programming:** You’d have to write out very specific instructions: “Go forward three steps, look for a red ball, pick it up with your mouth, return to me.”  It’s rigid and doesn\'t handle variations.\n* **Machine Learning:** Instead of giving detailed instructions, you show the dog *many* examples of "fetch." You throw the ball repeatedly, rewarding good behavior (bringing it back) and correcting mistakes. The dog *learns* what “fetch” means by observing patterns and adjusting its actions over time.\n\n**That’s exactly what machine learning does!** \n\nHere\'s a more formal breakdown:\n\n1. **Data is Key:** Machine learning algorithms need data t

> **Notice:** The model produces a fluent, contextually appropriate answer — without being explicitly programmed with facts about machine learning. That is the power of large-scale pre-training.

---
## 3 · System prompts: shaping the model's behaviour

So far we have used `llm.generate(prompt)`, which sends the prompt as a plain user message. But chat-based LLMs actually receive a **list of messages**, each with a `role`:

| Role | Who | Purpose |
|---|---|---|
| `"system"` | Developer | Sets the model's persona, constraints, and focus — invisible to the end user |
| `"user"` | Human | The message the user typed |
| `"assistant"` | Model | The model's previous replies (used in multi-turn chat) |

The **system prompt** is always the first message. It is one of the most important tools in practice: it is how every commercial chatbot (customer service bots, coding assistants, tutors) is given its personality and guardrails.

```python
messages = [
    {"role": "system",  "content": "You are a concise assistant. Answer in one sentence."},
    {"role": "user",    "content": "What is gradient descent?"},
]
reply = llm.chat(messages)
```

In [ ]:
# Provided — run this cell to see the effect of a system prompt
no_system = [
    {"role": "user", "content": "What is gradient descent?"}
]

with_system = [
    {"role": "system", "content": "You are a one-sentence explainer. Always answer in exactly one sentence."},
    {"role": "user",   "content": "What is gradient descent?"},
]

print("─── No system prompt ───")
print(llm.chat(no_system, temperature=0, max_new_tokens=150))
print()
print("─── With system prompt (one-sentence constraint) ───")
print(llm.chat(with_system, temperature=0, max_new_tokens=150))

### Exercise 2.1.2 `[Core]` — Same question, different personas

The same question receives very different answers depending on the system prompt. Ask the following question using three different system prompts and compare:

> *"Explain what a neural network is."*

Personas to try:
1. No system prompt (default behaviour — use `llm.generate`)
2. `"You are a friendly professor explaining concepts to first-year engineering students. Use analogies and avoid jargon."`
3. `"You are a senior ML engineer writing concise internal documentation. Be precise and technical."`

For each, use `llm.chat(messages, temperature=0)` and print the response. Notice how *style* changes while *facts* stay the same.

In [ ]:
# YOUR CODE HERE
# Hint: The same question receives very different answers depending on the system prompt. Ask the following question using th...
raise NotImplementedError("Complete this exercise")

> **Key observations:**
> - The *factual content* (what a neural network is) stays roughly the same — the model's underlying knowledge does not change.
> - The *style* changes dramatically: vocabulary, sentence length, use of analogies, depth of detail.
> - A system prompt is the main lever for **prompt engineering**: adapting behaviour without touching the model.
>
> **Critical limitation to notice now:** The system prompt can only draw on knowledge the model already has from training. If all three personas are asked *"What is TechRetail's return policy for electronics?"*, all three will still get it wrong — because no system prompt can inject facts the model was never trained on. That is a job for RAG (Lab 2.3).

---
## 4 · Temperature: creativity vs. consistency

LLM outputs are probabilistic — the model samples from a probability distribution over possible next tokens. **Temperature** controls how sharp or flat that distribution is:
- `temperature = 0` → always picks the most probable token → **deterministic, repeatable**.
- `temperature = 1` → samples proportionally to probabilities → **diverse, sometimes surprising**.
- `temperature > 1` → flattens the distribution → **very creative but also unreliable**.

### Exercise 2.1.3 `[Core]` — Temperature experiment

Run the same creative prompt three times:
- Once at `temperature=0` (greedy)
- Once at `temperature=0.7` (default — balanced)
- Once at `temperature=1.2` (high creativity)

Compare the outputs. Which temperature gives the most consistent answer? Which is the most surprising?

In [ ]:
# YOUR CODE HERE
# Hint: Run the same creative prompt three times:
raise NotImplementedError("Complete this exercise")

> **Key observation:** At `temperature=0` the model gives the same answer every run. At higher temperatures, outputs diverge — both more interesting and less predictable. For factual tasks (e.g., answering a customer question), keep temperature low. For creative tasks (brainstorming, writing), higher temperatures are useful.
>
> For the rest of this lab we use `temperature=0` when we want reproducible outputs to compare.

---
## 5 · The hallucination problem — live demo

This is the most important section of this lab. We are going to deliberately make the LLM hallucinate — and then understand *why* it happens, not as a bug, but as a structural property.

**Background:** TechRetail Co. is a fictional Colombian electronics retailer. Their return policy for electronics is **15 days** (not the usual 30 days). Their express shipping costs **39,900 COP**. These numbers are not in any public dataset or website — the LLM has never seen them.

### Exercise 2.1.4 `[Core]` — Trigger a hallucination

Ask the LLM the two questions below. Use `temperature=0` for reproducibility.
1. What is the return window for electronics at TechRetail Co.?
2. How much does express shipping cost at TechRetail Co.?

Observe what the model answers. Does it refuse? Does it make something up? Does it acknowledge uncertainty?

In [ ]:
# YOUR CODE HERE
# Hint: Ask the LLM the two questions below. Use temperature=0 for reproducibility.
raise NotImplementedError("Complete this exercise")

In [ ]:
q2 = "How much does express shipping cost at TechRetail Co.?"
print("Q:", q2)
print("A:", llm.generate(q2, temperature=0))

**Ground truth** (from the TechRetail policy documents):
- Electronics return window: **15 days** (not 30)
- Express shipping: **39,900 COP**

Compare with what the LLM said. Common behaviours:
- It gives a plausible-sounding but wrong number (hallucination).
- It hedges ('I don't have specific information about TechRetail') — better, but still unhelpful.
- It extrapolates from similar companies it saw during training — risky in production.

**Note:** You may notice that even using a system prompt that says *"be honest, do not guess"* does not prevent this — the model simply cannot know what it was never shown. That is the limit you observed at the end of Section 3.

### Why does this happen?

An LLM is trained to predict the next most likely token given everything before it. It has *no mechanism* to distinguish between:
- Things it actually knows from training data.
- Things that sound plausible based on learned patterns.

The model is optimised for fluency and coherence, not for truth. As Ji et al. (2023) put it: *hallucinations are not a bug that can be patched — they are a structural property of how these models work.*

**This is exactly why we need RAG:** instead of relying on the model's memory, we retrieve the correct information and inject it into the prompt. That is Lab 2.3.

---
## 6 · `[Extension]` Context windows and token counting

### Exercise 2.1.5 `[Extension]` — How many tokens is your prompt?

LLMs process text as **tokens**, not characters or words. A token is roughly 4 characters on average, but it varies ("ChatGPT" might be one token, while "supercalifragilistic" might be four).

Every model has a **context window** — the maximum number of tokens it can process at once (input + output together). If you exceed it, older parts of the conversation are silently dropped.

Use `llm.count_tokens(text)` to count the tokens in a few strings and build intuition.

In [ ]:
# YOUR CODE HERE
# Hint: LLMs process text as tokens, not characters or words. A token is roughly 4 characters on average, but it varies ("Cha...
raise NotImplementedError("Complete this exercise")

### How big is our knowledge base in tokens?

In [ ]:
# YOUR CODE HERE
# Hint: LLMs process text as tokens, not characters or words. A token is roughly 4 characters on average, but it varies ("Cha...
raise NotImplementedError("Complete this exercise")

---
## Summary

| What we learned | Key takeaway |
|---|---|
| LLMs generate text by predicting the next token | They are powerful pattern matchers, not fact databases |
| System prompts shape style and behaviour | The main lever for prompt engineering — but cannot inject new facts |
| Temperature controls sampling randomness | Low temp = consistent; high temp = creative |
| Hallucination is structural | The model cannot distinguish memory from plausible inference |
| Context windows are finite | Full KB injection is impractical at scale — RAG is the answer |

**Next → Lab 2.2:** We build a multi-turn chatbot with a system prompt and explore exactly how far prompt engineering alone takes us before RAG becomes necessary.